In [ ]:
# ============================================================
# DENGUE HOST–VIRUS DYNAMICS PROJECT
# NOTEBOOK 01 — DATA INGESTION AND AUDIT
# ============================================================

PROJECT_NAME = "Dengue Host–Virus Dynamics"
NOTEBOOK = "01_Data_Audit"
RANDOM_SEED = 10010

print("=" * 70)
print(PROJECT_NAME)
print("=" * 70)
print("Notebook :", NOTEBOOK)
print("Seed     :", RANDOM_SEED)

print("\nPrimary question:")
print(
    "Does temporal decoupling between dengue viral burden and "
    "host physiological recovery identify clinically meaningful "
    "disease states and predict subsequent deterioration?"
)

print("\nStatus: Notebook initialized successfully.")

In [ ]:
import os
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

np.random.seed(RANDOM_SEED)

print("=" * 70)
print("COMPUTATIONAL ENVIRONMENT")
print("=" * 70)

print("Python version :", sys.version.split()[0])
print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)
print("Operating system:", platform.system(), platform.release())

print("\nCurrent working directory:")
print(Path.cwd())

print("\nEnvironment initialized successfully.")

In [ ]:
DATA_FILENAME = "Longitudinal viremia and platelet 22May2024.csv"

candidate_locations = [
    Path.cwd() / DATA_FILENAME,
    Path.cwd().parent / DATA_FILENAME,
    Path.cwd() / "data" / DATA_FILENAME,
    Path.cwd().parent / "data" / DATA_FILENAME,
    Path.cwd().parent.parent / "data" / DATA_FILENAME,
]

DATA_PATH = None

for path in candidate_locations:
    if path.exists():
        DATA_PATH = path.resolve()
        break

print("=" * 70)
print("DATASET LOCATION CHECK")
print("=" * 70)

if DATA_PATH is None:
    print("Dataset not found automatically.")
    print("\nLocations checked:")
    for p in candidate_locations:
        print(" -", p)
else:
    print("Dataset found successfully:")
    print(DATA_PATH)

In [ ]:
assert DATA_PATH is not None, "Dataset path has not been resolved."

df_raw = pd.read_csv(DATA_PATH)

print("=" * 70)
print("RAW DATASET LOADED")
print("=" * 70)

print(f"Rows    : {df_raw.shape[0]:,}")
print(f"Columns : {df_raw.shape[1]:,}")

print("\nFirst 5 rows:")
display(df_raw.head())

In [ ]:
print("=" * 70)
print("COLUMN STRUCTURE CHECK")
print("=" * 70)

for i, col in enumerate(df_raw.columns, start=1):
    print(f"{i:02d}. {col}")

EXPECTED_COLUMNS = [
    "Study",
    "Code",
    "Age",
    "Sex",
    "Enrol_DOI",
    "Serotype",
    "Serology",
    "DOI",
    "PLT",
    "vir",
    "Severe_dengue",
    "doi_severe",
    "Leakage",
    "doi_leakage",
]

actual_columns = list(df_raw.columns)

missing_columns = [
    col for col in EXPECTED_COLUMNS
    if col not in actual_columns
]

unexpected_columns = [
    col for col in actual_columns
    if col not in EXPECTED_COLUMNS
]

print("\nMissing expected columns:")
print(missing_columns if missing_columns else "None")

print("\nUnexpected columns:")
print(unexpected_columns if unexpected_columns else "None")

print("\nExact expected column structure:",
      actual_columns == EXPECTED_COLUMNS)

print("\nDataset dimensions:")
print("Rows   :", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

In [ ]:
print("=" * 70)
print("DATA TYPE AND MISSINGNESS AUDIT")
print("=" * 70)

audit_types = pd.DataFrame({
    "Variable": df_raw.columns,
    "Data_Type": df_raw.dtypes.astype(str).values,
    "Non_Missing": df_raw.notna().sum().values,
    "Missing": df_raw.isna().sum().values,
})

audit_types["Missing_%"] = (
    100 * audit_types["Missing"] / len(df_raw)
).round(2)

display(audit_types)

print("\nSummary:")
print("Total rows :", len(df_raw))
print("Total cols :", df_raw.shape[1])
print("Variables with any missing data:",
      (audit_types["Missing"] > 0).sum())

In [ ]:
print("=" * 70)
print("PATIENT AND STUDY STRUCTURE")
print("=" * 70)

print("Unique Code values:")
print(df_raw["Code"].nunique(dropna=True))

print("\nUnique Study values:")
print(df_raw["Study"].nunique(dropna=True))

study_summary = (
    df_raw.groupby("Study", dropna=False)
          .agg(
              Rows=("Code", "size"),
              Unique_Codes=("Code", "nunique")
          )
          .reset_index()
          .sort_values("Unique_Codes", ascending=False)
)

print("\nStudy-level composition:")
display(study_summary)

print("\nTotal rows:", len(df_raw))
print("Total unique raw Code values:", df_raw["Code"].nunique())

In [ ]:
print("=" * 70)
print("CREATE SAFE PATIENT IDENTIFIER")
print("=" * 70)

df = df_raw.copy(deep=True)

df["Patient_ID"] = (
    df["Study"].astype(str).str.strip()
    + "__"
    + df["Code"].astype(str).str.strip()
)

print("Total rows:", len(df))
print("Unique raw Code values:", df["Code"].nunique(dropna=True))
print("Unique composite Patient_ID values:",
      df["Patient_ID"].nunique(dropna=True))

print("\nExample identifiers:")
display(
    df[["Study", "Code", "Patient_ID"]]
    .drop_duplicates()
    .head(10)
)

# Check whether the same Code occurs in more than one Study
code_study_count = (
    df.groupby("Code")["Study"]
      .nunique()
)

repeated_across_studies = code_study_count[code_study_count > 1]

print("\nCodes appearing in more than one Study:",
      len(repeated_across_studies))

if len(repeated_across_studies) > 0:
    print("\nExamples:")
    display(repeated_across_studies.head(20))
else:
    print("No raw patient codes are shared across studies.")

In [ ]:
print("=" * 70)
print("DUPLICATE PATIENT-DAY AUDIT")
print("=" * 70)

duplicate_mask = df.duplicated(
    subset=["Patient_ID", "DOI"],
    keep=False
)

n_duplicate_rows = int(duplicate_mask.sum())

duplicate_pairs = (
    df.loc[duplicate_mask, ["Patient_ID", "DOI"]]
      .drop_duplicates()
)

print("Rows involved in duplicate Patient_ID × DOI combinations:",
      n_duplicate_rows)

print("Number of duplicated Patient_ID × DOI pairs:",
      len(duplicate_pairs))

if n_duplicate_rows > 0:
    print("\nExamples of duplicated patient-day records:")
    display(
        df.loc[
            duplicate_mask,
            [
                "Study",
                "Code",
                "Patient_ID",
                "DOI",
                "PLT",
                "vir",
                "Severe_dengue",
                "Leakage"
            ]
        ]
        .sort_values(["Patient_ID", "DOI"])
        .head(50)
    )
else:
    print("\nNo duplicate patient-day records detected.")

In [ ]:
print("=" * 70)
print("LONGITUDINAL COVERAGE AUDIT")
print("=" * 70)

# Convert DOI safely for audit purposes
doi_numeric = pd.to_numeric(df["DOI"], errors="coerce")

print("DOI minimum :", doi_numeric.min())
print("DOI maximum :", doi_numeric.max())
print("Missing/non-numeric DOI values :", doi_numeric.isna().sum())

# Patient-level observation summary
patient_obs = (
    df.assign(DOI_numeric=doi_numeric)
      .groupby("Patient_ID")
      .agg(
          N_Observations=("DOI_numeric", "size"),
          First_DOI=("DOI_numeric", "min"),
          Last_DOI=("DOI_numeric", "max"),
          Platelet_Observations=("PLT", lambda x: x.notna().sum()),
          Viremia_Observations=("vir", lambda x: x.notna().sum())
      )
      .reset_index()
)

print("\nNumber of patients:", len(patient_obs))

print("\nObservation density per patient:")
display(
    patient_obs[
        [
            "N_Observations",
            "First_DOI",
            "Last_DOI",
            "Platelet_Observations",
            "Viremia_Observations"
        ]
    ].describe()
)

print("\nPatients by longitudinal depth:")
print("1 observation  :", (patient_obs["N_Observations"] == 1).sum())
print(">=2 observations:", (patient_obs["N_Observations"] >= 2).sum())
print(">=3 observations:", (patient_obs["N_Observations"] >= 3).sum())
print(">=4 observations:", (patient_obs["N_Observations"] >= 4).sum())
print(">=5 observations:", (patient_obs["N_Observations"] >= 5).sum())

print("\nDOI distribution:")
doi_distribution = (
    df.assign(DOI_numeric=doi_numeric)
      .groupby("DOI_numeric")
      .agg(
          Observations=("Patient_ID", "size"),
          Patients=("Patient_ID", "nunique")
      )
      .reset_index()
      .sort_values("DOI_numeric")
)

display(doi_distribution)

In [ ]:
print("=" * 70)
print("MISSINGNESS PROFILE")
print("=" * 70)

missingness = pd.DataFrame({
    "Variable": df.columns,
    "N_Missing": df.isna().sum().values,
    "N_NonMissing": df.notna().sum().values,
})

missingness["Missing_%"] = (
    100 * missingness["N_Missing"] / len(df)
).round(2)

missingness = missingness.sort_values(
    ["Missing_%", "Variable"],
    ascending=[False, True]
).reset_index(drop=True)

display(missingness)

print("\nVariables containing missing values:")
display(
    missingness.loc[
        missingness["N_Missing"] > 0,
        ["Variable", "N_Missing", "Missing_%"]
    ]
)

print("\nKey analysis variables:")
key_vars = [
    "Age",
    "Sex",
    "Serotype",
    "Serology",
    "DOI",
    "PLT",
    "vir",
    "Severe_dengue",
    "doi_severe",
    "Leakage",
    "doi_leakage"
]

display(
    missingness[
        missingness["Variable"].isin(key_vars)
    ].sort_values("Variable")
)

In [ ]:
print("=" * 70)
print("CATEGORICAL AND OUTCOME CODING AUDIT")
print("=" * 70)

categorical_vars = [
    "Sex",
    "Serotype",
    "Serology",
    "Severe_dengue",
    "Leakage"
]

for col in categorical_vars:
    print("\n" + "-" * 70)
    print(col)
    print("-" * 70)

    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="Rows")
    )

    counts["Percent"] = (
        100 * counts["Rows"] / len(df)
    ).round(2)

    display(counts)

In [ ]:
print("=" * 70)
print("PATIENT-LEVEL SEVERE DENGUE AND LEAKAGE AUDIT")
print("=" * 70)

def unique_nonmissing(series):
    return sorted(series.dropna().unique().tolist())

patient_outcomes_check = (
    df.groupby("Patient_ID")
      .agg(
          Severe_values=("Severe_dengue", unique_nonmissing),
          Leakage_values=("Leakage", unique_nonmissing),
          Severe_day_values=("doi_severe", unique_nonmissing),
          Leakage_day_values=("doi_leakage", unique_nonmissing)
      )
      .reset_index()
)

print("Total patients:", len(patient_outcomes_check))

print("\nSEVERE DENGUE PATTERNS")
print(
    patient_outcomes_check["Severe_values"]
    .astype(str)
    .value_counts(dropna=False)
)

print("\nLEAKAGE PATTERNS")
print(
    patient_outcomes_check["Leakage_values"]
    .astype(str)
    .value_counts(dropna=False)
)

print("\nPatients with >1 Severe_dengue value:",
      patient_outcomes_check["Severe_values"].apply(len).gt(1).sum())

print("Patients with >1 Leakage value:",
      patient_outcomes_check["Leakage_values"].apply(len).gt(1).sum())

print("\nPatients with recorded severe-dengue event day:",
      patient_outcomes_check["Severe_day_values"].apply(len).gt(0).sum())

print("Patients with recorded leakage event day:",
      patient_outcomes_check["Leakage_day_values"].apply(len).gt(0).sum())

In [ ]:
print("=" * 70)
print("EVENT TIMING INTEGRITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# Create patient-level outcome summary
# ------------------------------------------------------------

patient_event_audit = (
    df.groupby("Patient_ID")
      .agg(
          Severe_dengue=("Severe_dengue", "first"),
          Leakage=("Leakage", "first"),
          doi_severe=("doi_severe", "first"),
          doi_leakage=("doi_leakage", "first"),
          First_DOI=("DOI", "min"),
          Last_DOI=("DOI", "max"),
          Study=("Study", "first"),
          Serotype=("Serotype", "first"),
          Age=("Age", "first"),
          Sex=("Sex", "first")
      )
      .reset_index()
)

print("Patients:", len(patient_event_audit))

# ------------------------------------------------------------
# Severe dengue timing
# ------------------------------------------------------------

severe_positive = patient_event_audit[
    patient_event_audit["Severe_dengue"] == 1
].copy()

print("\n" + "-" * 70)
print("SEVERE DENGUE")
print("-" * 70)

print("Positive patients:", len(severe_positive))
print(
    "Positive patients with event day:",
    severe_positive["doi_severe"].notna().sum()
)

print(
    "Positive patients without event day:",
    severe_positive["doi_severe"].isna().sum()
)

print("\nSevere dengue event-day distribution:")
display(
    severe_positive["doi_severe"]
    .value_counts()
    .sort_index()
    .rename_axis("DOI_severe")
    .reset_index(name="Patients")
)

# ------------------------------------------------------------
# Leakage timing
# ------------------------------------------------------------

leakage_positive = patient_event_audit[
    patient_event_audit["Leakage"] == 1
].copy()

print("\n" + "-" * 70)
print("PLASMA LEAKAGE")
print("-" * 70)

print("Positive patients:", len(leakage_positive))
print(
    "Positive patients with event day:",
    leakage_positive["doi_leakage"].notna().sum()
)

print(
    "Positive patients without event day:",
    leakage_positive["doi_leakage"].isna().sum()
)

print("\nLeakage event-day distribution:")
display(
    leakage_positive["doi_leakage"]
    .value_counts()
    .sort_index()
    .rename_axis("DOI_leakage")
    .reset_index(name="Patients")
)

# ------------------------------------------------------------
# Check impossible/conflicting timing
# ------------------------------------------------------------

severe_before_observation = severe_positive[
    severe_positive["doi_severe"] <
    severe_positive["First_DOI"]
]

leakage_before_observation = leakage_positive[
    leakage_positive["doi_leakage"].notna() &
    (
        leakage_positive["doi_leakage"] <
        leakage_positive["First_DOI"]
    )
]

print("\n" + "-" * 70)
print("TIMING FLAGS")
print("-" * 70)

print(
    "Severe events before first observed DOI:",
    len(severe_before_observation)
)

print(
    "Leakage events before first observed DOI:",
    len(leakage_before_observation)
)

print("\nLeakage-positive patients lacking event timing:")
display(
    leakage_positive[
        leakage_positive["doi_leakage"].isna()
    ][
        [
            "Patient_ID",
            "Study",
            "Serotype",
            "Age",
            "Sex",
            "First_DOI",
            "Last_DOI"
        ]
    ]
)

In [ ]:
print("=" * 70)
print("LANDMARK FEASIBILITY AUDIT")
print("=" * 70)

landmark_df = patient_event_audit.copy()

# Ensure event days are numeric
landmark_df["doi_severe_num"] = pd.to_numeric(
    landmark_df["doi_severe"], errors="coerce"
)

landmark_df["doi_leakage_num"] = pd.to_numeric(
    landmark_df["doi_leakage"], errors="coerce"
)

results = []

for landmark in [3, 4, 5]:

    # Patient must have entered observation by the landmark
    observed = landmark_df["First_DOI"] <= landmark

    # ========================================================
    # SEVERE DENGUE
    # ========================================================

    severe_already = (
        observed
        & (landmark_df["Severe_dengue"] == 1)
        & (landmark_df["doi_severe_num"] <= landmark)
    )

    severe_future = (
        observed
        & (landmark_df["Severe_dengue"] == 1)
        & (landmark_df["doi_severe_num"] > landmark)
    )

    severe_none = (
        observed
        & (landmark_df["Severe_dengue"] == 0)
    )

    severe_analysis_cohort = severe_future | severe_none

    severe_n = severe_analysis_cohort.sum()
    severe_events = severe_future.sum()

    severe_event_rate = (
        100 * severe_events / severe_n
        if severe_n > 0 else np.nan
    )

    # ========================================================
    # PLASMA LEAKAGE
    # ========================================================

    leakage_missing_label = (
        observed
        & landmark_df["Leakage"].isna()
    )

    leakage_unknown_timing = (
        observed
        & (landmark_df["Leakage"] == 1)
        & landmark_df["doi_leakage_num"].isna()
    )

    leakage_already = (
        observed
        & (landmark_df["Leakage"] == 1)
        & landmark_df["doi_leakage_num"].notna()
        & (landmark_df["doi_leakage_num"] <= landmark)
    )

    leakage_future = (
        observed
        & (landmark_df["Leakage"] == 1)
        & landmark_df["doi_leakage_num"].notna()
        & (landmark_df["doi_leakage_num"] > landmark)
    )

    leakage_none = (
        observed
        & (landmark_df["Leakage"] == 0)
    )

    leakage_analysis_cohort = leakage_future | leakage_none

    leakage_n = leakage_analysis_cohort.sum()
    leakage_events = leakage_future.sum()

    leakage_event_rate = (
        100 * leakage_events / leakage_n
        if leakage_n > 0 else np.nan
    )

    # ========================================================
    # STORE RESULTS
    # ========================================================

    results.append({
        "Landmark_DOI": landmark,

        "Patients_observed_by_landmark":
            int(observed.sum()),

        "Severe_already_by_landmark":
            int(severe_already.sum()),

        "Future_severe_events":
            int(severe_events),

        "Severe_analysis_N":
            int(severe_n),

        "Future_severe_rate_%":
            round(severe_event_rate, 2),

        "Leakage_already_by_landmark":
            int(leakage_already.sum()),

        "Future_leakage_events":
            int(leakage_events),

        "Leakage_analysis_N":
            int(leakage_n),

        "Future_leakage_rate_%":
            round(leakage_event_rate, 2),

        "Leakage_positive_unknown_timing":
            int(leakage_unknown_timing.sum()),

        "Leakage_missing_label":
            int(leakage_missing_label.sum())
    })


landmark_summary = pd.DataFrame(results)

display(landmark_summary)

In [ ]:
print("=" * 70)
print("PREDICTOR AVAILABILITY BY LANDMARK")
print("=" * 70)

df_work = df.copy()

df_work["DOI_num"] = pd.to_numeric(
    df_work["DOI"],
    errors="coerce"
)

availability_results = []

for landmark in [3, 4, 5]:

    # Restrict to observations available up to landmark only
    temp = df_work[
        df_work["DOI_num"] <= landmark
    ].copy()

    patient_availability = (
        temp.groupby("Patient_ID")
            .agg(
                First_DOI=("DOI_num", "min"),
                Last_DOI_by_landmark=("DOI_num", "max"),

                N_rows_by_landmark=("DOI_num", "size"),

                N_PLT_by_landmark=(
                    "PLT",
                    lambda x: x.notna().sum()
                ),

                N_VIR_by_landmark=(
                    "vir",
                    lambda x: x.notna().sum()
                )
            )
            .reset_index()
    )

    n_patients = len(patient_availability)

    both_any = (
        (patient_availability["N_PLT_by_landmark"] >= 1)
        &
        (patient_availability["N_VIR_by_landmark"] >= 1)
    ).sum()

    both_two = (
        (patient_availability["N_PLT_by_landmark"] >= 2)
        &
        (patient_availability["N_VIR_by_landmark"] >= 2)
    ).sum()

    both_three = (
        (patient_availability["N_PLT_by_landmark"] >= 3)
        &
        (patient_availability["N_VIR_by_landmark"] >= 3)
    ).sum()

    availability_results.append({
        "Landmark_DOI": landmark,
        "Patients_observed": n_patients,

        ">=1_PLT_and_>=1_VIR": int(both_any),
        "Pct_with_both_any":
            round(100 * both_any / n_patients, 2),

        ">=2_PLT_and_>=2_VIR": int(both_two),
        "Pct_with_both_2":
            round(100 * both_two / n_patients, 2),

        ">=3_PLT_and_>=3_VIR": int(both_three),
        "Pct_with_both_3":
            round(100 * both_three / n_patients, 2)
    })

availability_summary = pd.DataFrame(
    availability_results
)

display(availability_summary)

In [ ]:
print("=" * 70)
print("DOI-4 PRIMARY LANDMARK COHORT DEFINITION")
print("=" * 70)

PRIMARY_LANDMARK = 4

cohort_doi4 = patient_event_audit.copy()

# Ensure numeric event timing
cohort_doi4["doi_severe_num"] = pd.to_numeric(
    cohort_doi4["doi_severe"],
    errors="coerce"
)

cohort_doi4["doi_leakage_num"] = pd.to_numeric(
    cohort_doi4["doi_leakage"],
    errors="coerce"
)

# ============================================================
# 1. Must have entered observation by DOI 4
# ============================================================

cohort_doi4["Observed_by_DOI4"] = (
    cohort_doi4["First_DOI"] <= PRIMARY_LANDMARK
)

# ============================================================
# 2. SEVERE DENGUE OUTCOME
# ============================================================

cohort_doi4["Severe_already_by_DOI4"] = (
    (cohort_doi4["Severe_dengue"] == 1)
    &
    (cohort_doi4["doi_severe_num"] <= PRIMARY_LANDMARK)
)

cohort_doi4["Future_severe_after_DOI4"] = (
    (cohort_doi4["Severe_dengue"] == 1)
    &
    (cohort_doi4["doi_severe_num"] > PRIMARY_LANDMARK)
)

cohort_doi4["Eligible_severe_prediction"] = (
    cohort_doi4["Observed_by_DOI4"]
    &
    ~cohort_doi4["Severe_already_by_DOI4"]
)

# Final binary outcome for eligible patients
cohort_doi4["Severe_future_outcome"] = np.where(
    cohort_doi4["Eligible_severe_prediction"],
    cohort_doi4["Future_severe_after_DOI4"].astype(int),
    np.nan
)

# ============================================================
# 3. PLASMA LEAKAGE OUTCOME
# ============================================================

cohort_doi4["Leakage_missing_label"] = (
    cohort_doi4["Leakage"].isna()
)

cohort_doi4["Leakage_positive_unknown_timing"] = (
    (cohort_doi4["Leakage"] == 1)
    &
    cohort_doi4["doi_leakage_num"].isna()
)

cohort_doi4["Leakage_already_by_DOI4"] = (
    (cohort_doi4["Leakage"] == 1)
    &
    cohort_doi4["doi_leakage_num"].notna()
    &
    (cohort_doi4["doi_leakage_num"] <= PRIMARY_LANDMARK)
)

cohort_doi4["Future_leakage_after_DOI4"] = (
    (cohort_doi4["Leakage"] == 1)
    &
    cohort_doi4["doi_leakage_num"].notna()
    &
    (cohort_doi4["doi_leakage_num"] > PRIMARY_LANDMARK)
)

cohort_doi4["Eligible_leakage_prediction"] = (
    cohort_doi4["Observed_by_DOI4"]
    &
    ~cohort_doi4["Leakage_missing_label"]
    &
    ~cohort_doi4["Leakage_positive_unknown_timing"]
    &
    ~cohort_doi4["Leakage_already_by_DOI4"]
)

cohort_doi4["Leakage_future_outcome"] = np.where(
    cohort_doi4["Eligible_leakage_prediction"],
    cohort_doi4["Future_leakage_after_DOI4"].astype(int),
    np.nan
)

# ============================================================
# 4. SUMMARY
# ============================================================

print("\nPATIENTS OBSERVED BY DOI 4")
print(
    cohort_doi4["Observed_by_DOI4"].sum()
)

print("\nSEVERE DENGUE")
print(
    "Already severe by DOI 4:",
    cohort_doi4["Severe_already_by_DOI4"].sum()
)

print(
    "Eligible prediction cohort:",
    cohort_doi4["Eligible_severe_prediction"].sum()
)

print(
    "Future severe events:",
    cohort_doi4["Future_severe_after_DOI4"].sum()
)

print(
    "Future severe event rate:",
    round(
        100 *
        cohort_doi4.loc[
            cohort_doi4["Eligible_severe_prediction"],
            "Severe_future_outcome"
        ].mean(),
        2
    ),
    "%"
)

print("\nPLASMA LEAKAGE")
print(
    "Already leakage by DOI 4:",
    cohort_doi4["Leakage_already_by_DOI4"].sum()
)

print(
    "Positive but unknown event timing:",
    cohort_doi4["Leakage_positive_unknown_timing"].sum()
)

print(
    "Missing leakage classification:",
    cohort_doi4["Leakage_missing_label"].sum()
)

print(
    "Eligible prediction cohort:",
    cohort_doi4["Eligible_leakage_prediction"].sum()
)

print(
    "Future leakage events:",
    cohort_doi4["Future_leakage_after_DOI4"].sum()
)

print(
    "Future leakage event rate:",
    round(
        100 *
        cohort_doi4.loc[
            cohort_doi4["Eligible_leakage_prediction"],
            "Leakage_future_outcome"
        ].mean(),
        2
    ),
    "%"
)

# ============================================================
# 5. INTERNAL CONSISTENCY ASSERTIONS
# ============================================================

assert cohort_doi4["Observed_by_DOI4"].sum() == 2390
assert cohort_doi4["Severe_already_by_DOI4"].sum() == 15
assert cohort_doi4["Future_severe_after_DOI4"].sum() == 50
assert cohort_doi4["Eligible_severe_prediction"].sum() == 2375

assert cohort_doi4["Leakage_already_by_DOI4"].sum() == 75
assert cohort_doi4["Future_leakage_after_DOI4"].sum() == 255
assert cohort_doi4["Leakage_positive_unknown_timing"].sum() == 26
assert cohort_doi4["Eligible_leakage_prediction"].sum() == 2238

print("\nAll DOI-4 cohort integrity checks passed.")

In [ ]:
print("=" * 70)
print("NOTEBOOK 01 — FINAL FROZEN AUDIT SUMMARY")
print("=" * 70)

from pathlib import Path
import json

# ============================================================
# CREATE OUTPUT DIRECTORIES
# ============================================================

PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
AUDIT_DIR = OUTPUT_DIR / "audit"

OUTPUT_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)
AUDIT_DIR.mkdir(exist_ok=True)

# ============================================================
# FROZEN STUDY CONSTANTS
# ============================================================

audit_final = {
    "project": PROJECT_NAME,
    "notebook": NOTEBOOK,
    "random_seed": int(RANDOM_SEED),

    "raw_rows": int(df_raw.shape[0]),
    "raw_columns": int(df_raw.shape[1]),
    "unique_patients": int(df["Patient_ID"].nunique()),

    "duplicate_patient_day_rows": int(n_duplicate_rows),

    "primary_landmark_DOI": 4,

    "patients_observed_by_DOI4":
        int(cohort_doi4["Observed_by_DOI4"].sum()),

    "severe_already_by_DOI4":
        int(cohort_doi4["Severe_already_by_DOI4"].sum()),

    "severe_prediction_N":
        int(cohort_doi4["Eligible_severe_prediction"].sum()),

    "future_severe_events":
        int(cohort_doi4["Future_severe_after_DOI4"].sum()),

    "future_severe_event_rate_percent":
        round(
            100 *
            cohort_doi4.loc[
                cohort_doi4["Eligible_severe_prediction"],
                "Severe_future_outcome"
            ].mean(),
            2
        ),

    "leakage_already_by_DOI4":
        int(cohort_doi4["Leakage_already_by_DOI4"].sum()),

    "leakage_unknown_timing":
        int(
            cohort_doi4[
                "Leakage_positive_unknown_timing"
            ].sum()
        ),

    "leakage_missing_classification":
        int(cohort_doi4["Leakage_missing_label"].sum()),

    "leakage_prediction_N":
        int(cohort_doi4["Eligible_leakage_prediction"].sum()),

    "future_leakage_events":
        int(cohort_doi4["Future_leakage_after_DOI4"].sum()),

    "future_leakage_event_rate_percent":
        round(
            100 *
            cohort_doi4.loc[
                cohort_doi4["Eligible_leakage_prediction"],
                "Leakage_future_outcome"
            ].mean(),
            2
        )
}

# ============================================================
# DISPLAY
# ============================================================

for key, value in audit_final.items():
    print(f"{key:40s}: {value}")

# ============================================================
# EXPORT AUDIT SUMMARY
# ============================================================

json_path = AUDIT_DIR / "Notebook01_Frozen_Audit_Summary.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        audit_final,
        f,
        indent=4
    )

# ============================================================
# EXPORT SUPPORTING TABLES
# ============================================================

missingness.to_csv(
    TABLE_DIR / "Table_Audit_Missingness.csv",
    index=False
)

landmark_summary.to_csv(
    TABLE_DIR / "Table_Audit_Landmark_Feasibility.csv",
    index=False
)

availability_summary.to_csv(
    TABLE_DIR / "Table_Audit_Predictor_Availability.csv",
    index=False
)

# Save the DOI-4 patient-level cohort definition
cohort_doi4.to_csv(
    AUDIT_DIR / "DOI4_Patient_Level_Cohort_Definition.csv",
    index=False
)

print("\n" + "=" * 70)
print("EXPORT COMPLETE")
print("=" * 70)

print("Audit summary:")
print(json_path)

print("\nPatient-level DOI-4 cohort:")
print(
    AUDIT_DIR /
    "DOI4_Patient_Level_Cohort_Definition.csv"
)

print("\nNotebook 01 status: COMPLETE AND FROZEN")